# My Day 5 - Introduction to Agents!

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [2]:
load_dotenv(override=True)

groq_api = os.getenv("GROQ_API_KEY")

if not groq_api:
    print("Groq API not found\n")
else:
    print("Groq API found and looking good so far!\n")

groq_base = "https://api.groq.com/openai/v1"
model_groq = "meta-llama/llama-4-scout-17b-16e-instruct"

DB = "prices_.db"

openai = OpenAI()
groq = OpenAI(base_url=groq_base, api_key=groq_api)

Groq API found and looking good so far!



In [3]:
system_prompt = """
You are a helpful assistant for an airline called FlightAI.
Give short, courteous and precise answers, with no more than 1 sentence.
If you didn't know a ticket price for an unknown city asked by the customer,\
kindly ask the customers if they heard any price around in order for you \
to add it into the database.
If you don't know the answer, please say so.
If someone ask you about dates, you say that you don't know prices for that day.
"""

In [4]:
def get_ticket_price(city):
    with sqlite3.connect(DB) as connection:
        cursor = connection.cursor()
        cursor.execute('SELECT price FROM prices_ WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Prices for {city} ---> {result[0]}" if result else "Unknown Ticket Price"

In [5]:
get_ticket_price("Madrid")

'Prices for Madrid ---> 580'

In [6]:
import re
def normalize_price_to_int(value) -> int:
    if isinstance(value, int):
        return value
    if isinstance(value, float):
        return int(value)  # truncates decimals

    s = str(value).strip().replace("$", "").replace(",", "")
    m = re.search(r"-?\d+(?:\.\d+)?", s)  # supports "1240", "1240.99", "$1,240.99"
    if not m:
        raise ValueError(f"Invalid price: {value}")
    return int(float(m.group(0)))  # -> int

In [7]:
def insert_ticket_values(city, price):
    price = normalize_price_to_int(price)
    with sqlite3.connect(DB) as connection:
        cursor = connection.cursor()
        cursor.execute('INSERT INTO prices_ (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.strip().lower(), price, price))
        connection.commit()
    return f"Price saved for {city} ---> {price}"

In [106]:
def list_destinations():
    with sqlite3.connect(DB) as connection:
        cursor = connection.cursor()
        cursor.execute('SELECT city FROM prices_')
        rows = cursor.fetchall()
        return [row[0].title() for row in rows]

In [107]:
list_destinations()

['Caracas', 'Dubai', 'London', 'Lyon', 'Madrid', 'New York', 'Paris', 'Tokyo']

In [40]:
def get_cheapest_destinations():
    with sqlite3.connect(DB) as connection:
        cursor = connection.cursor()
        cursor.execute('SELECT city, price '
                       'FROM prices_ '
                       'ORDER BY price ASC '
                       'LIMIT 5;')
        result = cursor.fetchall()
    return result

In [50]:
get_cheapest_destinations()

[('madrid', 580),
 ('dubai', 650),
 ('paris', 655),
 ('lyon', 770),
 ('london', 800)]

In [75]:
def get_price_range(min_range, max_range):
    min_range, max_range = int(min_range), int(max_range)
    with sqlite3.connect(DB) as connection:
        cursor = connection.cursor()
        cursor.execute('SELECT city, price '
                       'FROM prices_ '
                       'WHERE price BETWEEN ? AND ?;', (min_range, max_range))
        result = cursor.fetchall()
    return result

In [60]:
get_price_range(900, 1500)

[('new york', 975), ('tokyo', 1240)]

In [113]:
def get_price_stats():
    with sqlite3.connect(DB) as connection:
        cursor = connection.cursor()
        cursor.execute('SELECT MIN(price) as minimum, MAX(price) as maximum, AVG(price) as average, COUNT(*) as howManyFlights '
                       'FROM prices_ ')
        result = cursor.fetchall()
    return f"Minimum: {result[0][0]}, Maximum: {result[0][1]}, Average: {int(result[0][2])}, Count: {result[0][3]}"

In [66]:
get_price_stats()

'Minimum: 580, Maximum: 1240, Average: 815.0, Count: 8'

In [8]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments or "{}")
        if tool_call.function.name == "get_ticket_price":
            city = arguments.get("city") or arguments.get("destination")
            result = get_ticket_price(city)
        elif tool_call.function.name == "insert_ticket_values":
            city = arguments.get("city") or arguments.get("destination")
            price = arguments.get("price")
            result = insert_ticket_values(city, price)
        else:
            continue
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": str(tool_call.id)
        })
    return responses

In [93]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a ticket to a destination city",
    "parameters": {
        "type": "object",
        "properties": {
            "city":{
                "type": "string",
                "description": "The city where the customer wants to travel to."
            },
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "insert_ticket_values",
    "description": "Insert cities and their respective flight ticket price into the database.",
    "parameters": {
        "type": "object",
        "properties": {
            "city":{
                "type": "string",
                "description": "The new city to be inserted into the database."
            },
            "price":{
                "type": "string",
                "description": "Price for tickets to travel to the city."
            }
        },
        "required": ["city", "price"],
        "additionalProperties": False
    }
}

list_destination_function = {
    "name": "list_destinations",
    "description": "Gives list of available cities with ticket prices to the customer.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

cheapest_destinations_function = {
    "name": "get_cheapest_destinations",
    "description": "Gives list of cheapest registered destinations to the customer.",
     "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

price_range_function = {
    "name": "get_price_range",
    "description": "Gives a list of flights that goes into a price scope given by the customer.",
    "parameters": {
        "type": "object",
        "properties": {
            "min_range": {
                "type": "string",
                "description": "Minimum number adopted inside the wanted scope of prices."
            },
            "max_range": {
                "type": "string",
                "description": "Maximum number adopted inside the wanted scope of prices."
            },
        },
        "required": ["min_range", "max_range"],
        "additionalProperties": False
    }
}

price_stats_function = {
    "name": "get_price_stats",
    "description": "Gives the stats of all the destination ticket flight prices recorded in the system database",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

In [94]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function},
    {"type": "function", "function": list_destination_function},
    {"type": "function", "function": cheapest_destinations_function},
    {"type": "function", "function": price_range_function},
    {"type": "function", "function": price_stats_function},
]

In [89]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=model_groq, messages=messages, tools=tools)

    #To make many tool calls in a single message:
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        #for message in messages:
        #    print(message)
        response = groq.chat.completions.create(model=model_groq, messages=messages, tools=tools)

    return response.choices[0].message.content

In [90]:
gr.ChatInterface(fn=chat, type='messages').launch(debug=True, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


## Building Multi-Modal

Some imports for handling images

In [13]:
import base64
from io import BytesIO
from PIL import Image

#### IMPORTANT: openai charges 4 cents each time it generates images

In [14]:
def artist(city):
    image_response = openai.images.generate(
        model="dall-e-3",
        prompt=f"An image representing a vacation in {city}, showing tourists spots and everything unique about {city},"
               f"in a vibrant pop-art style.",
        size="1024x1024",
        n=1,
        response_format="b64_json"
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
#image = artist("New York City")
#display(image)

#### Generating audios

In [15]:
def talker(message):
    response = openai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=message
    )
    return response.content

## Let's do this:

1) A multi-modal AI assistant capable of generate images and voice.
2) Tool calling with database lookup.
3) A step towards Agentic workflow.

In [101]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history
    response = groq.chat.completions.create(model=model_groq, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=model_groq, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])

    return history, voice, image

In [102]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments or "{}")
        if tool_call.function.name == "get_ticket_price":
            city = arguments.get("city") or arguments.get("destination")
            cities.append(city)
            result = get_ticket_price(city)
        elif tool_call.function.name == "insert_ticket_values":
            city = arguments.get("city") or arguments.get("destination")
            price = arguments.get("price")
            result = insert_ticket_values(city, price)
        elif tool_call.function.name == "list_destinations":
            result = list_destinations()
        elif tool_call.function.name == "get_cheapest_destinations":
            result = get_cheapest_destinations()
        elif tool_call.function.name == "get_price_range":
            min_range = float(arguments.get("min_range"))
            max_range = float(arguments.get("max_range"))
            result = get_price_range(min_range, max_range)
        elif tool_call.function.name == "get_price_stats":
            result = get_price_stats()
        else:
            continue
        responses.append({
            "role": "tool",
            "content": result if isinstance(result, str) else json.dumps(result, ensure_ascii=False),
            "tool_call_id": str(tool_call.id)
        })
    return responses, cities

### Let's try with Blocks on Gradio

Allows us to control components on the callback

In [ ]:
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content":message}]

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type='messages')
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(interactive=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant")

    message.submit(fn=put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(fn=chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output])

ui.launch(inbrowser=True, auth=("juan", "ron"), debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
